In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import init_empty_weights
from accelerate.utils import load_fsdp_model

# 1. Load the base model architecture with the correct configuration
model_name = "your-base-model-name"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# It's recommended to initialize with empty weights to save memory
with init_empty_weights():
    model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. & 3. Load the FSDP sharded checkpoint into the model
# The `load_fsdp_model` function can be found in some community scripts or
# can be implemented by iterating through the sharded files and loading them.
# A simpler approach is to use a script that consolidates the checkpoint first.

# Assuming you have a consolidated state_dict
# state_dict = torch.load("path/to/consolidated_checkpoint.pt")
# model.load_state_dict(state_dict)

# # 4. Save the model in the standard Hugging Face format
# output_dir = "path/to/your/converted/hf/model"
# model.save_pretrained(output_dir)
# tokenizer.save_pretrained(output_dir)

In [2]:
from accelerate.utils import merge_fsdp_weights

# The directory where your FSDP checkpoint is saved
fsdp_checkpoint_path = "/code/hf_trainer_edu/checkpoint-122071/pytorch_model_fsdp_0"

# The path where you want to save the converted Hugging Face model
huggingface_model_path = "/code/rmt_model"

merge_fsdp_weights(
    checkpoint_dir=fsdp_checkpoint_path,
    output_path=huggingface_model_path,
    safe_serialization=True,  # Recommended to use safetensors
)

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import init_empty_weights
from accelerate.utils import load_fsdp_model
from train_gym.rmt.rmt_wrappers import (
    MemoryCell,
    RecurrentWrapper,
    MemoryCellTrain,
    RecurrentWrapperTrain,
    MemoryCellTrainLiger,
    lce_forward,
)

# 1. Load the base model architecture with the correct configuration
model_name = "unsloth/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# It's recommended to initialize with empty weights to save memory
with init_empty_weights():
    model = AutoModelForCausalLM.from_pretrained(model_name)
    cell = MemoryCellTrain(
        model,
        num_mem_tokens=32,
    )
    model = RecurrentWrapperTrain(
        cell,
        segment_size=1024,
        max_n_segments=2,
        vary_n_segments=False,
        k2=-1,
    )

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Some weights of LlamaForCausalLM were not initialized from the model checkpoint at unsloth/Llama-3.2-1B-Instruct and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [5]:
from safetensors.torch import load_file
import torch

file_path = "/code/rmt_model/model.safetensors"


state_dict = load_file(file_path, device="cpu")

In [19]:
model.load_state_dict(state_dict)
None

In [ ]:
# model = model.to('cuda:0')
model = model.to_empty(device="cuda:0")
None

RecurrentWrapperTrain(
  (memory_cell): MemoryCellTrain(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
              (k_proj): Linear(in_features=2048, out_features=512, bias=False)
              (v_proj): Linear(in_features=2048, out_features=512, bias=False)
              (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
            )
            (mlp): LlamaMLP(
              (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
              (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
              (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
              (act_fn): SiLUActivation()
            )
            (input_layernorm): LlamaRMSNo

In [15]:
model.memory_cell.model.device

device(type='cuda', index=0)

In [46]:
input_text = "the capital of russia is"
# input_text = "Transformers have recently emerged as a powerful tool for learning visual representations. In this paper, we identify and characterize artifacts in feature maps of both supervised and self-supervise"
with torch.no_grad():
    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
    ).to(model.memory_cell.model.device)
    result = model.generate(
        input_ids=input_ids["input_ids"],
        attention_mask=input_ids["attention_mask"],
        do_sample=False,
        max_new_tokens=20,
    )
    print(tokenizer.batch_decode(result))

[' the largest city of the country in the country of the country of the eastern Europe, and the eastern']


In [49]:
out, new_memory_state = model.memory_cell(input_ids=input_ids["input_ids"])

In [53]:
new_memory_state[:, :new_memory_state.shape[1] // 2, :].shape

torch.Size([1, 16, 2048])

In [3]:
import torch
torch.randn((1, 36, 2048))[:, :-4, :].shape

torch.Size([1, 32, 2048])

In [26]:
input_ids

{'input_ids': tensor([[128000,   1820,   6864,    315,  64766,    374]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [ ]:
tokenizer.batch_decode(input_ids["input_ids"])

['<|begin_of_text|>the capital of america is']

In [42]:
def greedy_decode(model, tokenizer, prompt, max_new_tokens=50, device="cpu"):

    print(f"Starting generation with prompt: '{prompt}'")

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    model.eval()
    # Generation loop
    for i in range(max_new_tokens):

        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits

        last_token_logits = logits[:, -1, :]
        next_token_id = torch.argmax(last_token_logits, dim=-1)

        input_ids = torch.cat([input_ids, next_token_id.unsqueeze(0)], dim=-1)

        if next_token_id.item() == tokenizer.eos_token_id:
            print("\nEOS token generated. Stopping.")
            break

        new_token_text = tokenizer.decode(next_token_id, skip_special_tokens=False)
        print(new_token_text, end="", flush=True)

    print("\n\n--- Finished Generation ---")

    generated_text = tokenizer.decode(input_ids[0])
    return generated_text


# input_text = "what is the capital of usa"
# input_text = "what is the capital of russia"
input_text = "1*2=2, 2*2=4, 3*2=6, 4*2="
final_output = greedy_decode(
    model=model,
    tokenizer=tokenizer,
    prompt=input_text,
    max_new_tokens=20,
    device=model.memory_cell.model.device,
)

# Print the final result
print("\nFinal Output:")
print(final_output)

Starting generation with prompt: '1*2=2, 2*2=4, 3*2=6, 4*2='
 3

*3*3*3*3*4*5*5*5*6

--- Finished Generation ---

Final Output:
<|begin_of_text|>1*2=2, 2*2=4, 3*2=6, 4*2= 3*3*3*3*3*4*5*5*5*6
